# Phase 2 - MLflow Tracking, Model Comparison and Model Registry

## Project Goal
This notebook implements the full Phase 2 workflow for the **Medical Insurance Charges** regression task.

The notebook includes:
- local MLflow experiment tracking,
- multiple model runs,
- logging of parameters, metrics, artifacts, and tags,
- best-model selection,
- model registration in the MLflow Model Registry,
- bonus confidence intervals with `statsmodels`,
- verification that the registered model can be loaded again.

## Dataset
The dataset file used here is:

- `insurance.csv`

## Required local setup
Before running this notebook, start a local MLflow tracking server in a separate terminal.

### Recommended command
```bash
mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlruns
```

Then open:

```text
http://127.0.0.1:5000
```

Keep that terminal open while running the notebook.


In [3]:
import warnings
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from mlflow.tracking import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'mlflow'

## Load Dataset

The dataset is loaded from the project root directory.
A folder for Phase 2 artifacts is also created.


In [ ]:
BASE_DIR = Path(".")
DATA_PATH = BASE_DIR / "insurance.csv"
OUTPUT_DIR = BASE_DIR / "phase2_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

## Initial Inspection

This section checks data types, missing values, and summary statistics.


In [ ]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDescriptive statistics:")
display(df.describe(include="all"))

## Preprocessing and Train-Test Split

The target variable is `charges`.

Preprocessing steps:
- numerical features are standardized with `StandardScaler`,
- categorical features are one-hot encoded with `OneHotEncoder`,
- the data is split into train and test sets using an 80/20 split.


In [ ]:
target_col = "charges"

X = df.drop(columns=[target_col])
y = df[target_col].astype(float)

numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", ohe, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

## MLflow Setup

All runs are logged to a local MLflow tracking server.
The experiment is stored under one consistent experiment name.


In [ ]:
TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "Insurance_Charges_Regression"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI :", TRACKING_URI)
print("Experiment   :", EXPERIMENT_NAME)

## Helper Functions

The following helper functions:
- compute metrics,
- save plots,
- build pipelines,
- run one complete MLflow experiment,
- and return a summary row for model comparison.


In [ ]:
def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
    }


def save_residual_plot(y_true, y_pred, output_path):
    residuals = y_true - y_pred
    plt.figure(figsize=(8, 4))
    plt.hist(residuals, bins=30, edgecolor="white")
    plt.title("Residual Distribution")
    plt.xlabel("Residual")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def save_pred_vs_actual_plot(y_true, y_pred, output_path):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.6)

    low = min(np.min(y_true), np.min(y_pred))
    high = max(np.max(y_true), np.max(y_pred))
    plt.plot([low, high], [low, high])

    plt.title("Predicted vs Actual")
    plt.xlabel("Actual Charges")
    plt.ylabel("Predicted Charges")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def build_pipeline(model):
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )


def run_experiment(run_name, model, params, dataset_version="v1.0"):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("algorithm", model.__class__.__name__)
        mlflow.set_tag("dataset_version", dataset_version)

        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("random_state", 42)
        mlflow.log_param("feature_count_raw", X_train.shape[1])

        for key, value in params.items():
            mlflow.log_param(key, value)

        pipeline = build_pipeline(model)
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        metrics = compute_metrics(y_test, y_pred)
        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        run_dir = OUTPUT_DIR / run.info.run_id
        run_dir.mkdir(parents=True, exist_ok=True)

        residual_plot_path = run_dir / "residuals.png"
        scatter_plot_path = run_dir / "pred_vs_actual.png"
        predictions_path = run_dir / "predictions.csv"

        save_residual_plot(y_test, y_pred, residual_plot_path)
        save_pred_vs_actual_plot(y_test, y_pred, scatter_plot_path)

        pred_df = pd.DataFrame({
            "actual": y_test.values,
            "predicted": y_pred
        })
        pred_df.to_csv(predictions_path, index=False)

        mlflow.log_artifact(str(residual_plot_path), artifact_path="plots")
        mlflow.log_artifact(str(scatter_plot_path), artifact_path="plots")
        mlflow.log_artifact(str(predictions_path), artifact_path="data")

        mlflow.sklearn.log_model(pipeline, artifact_path="model")

        return {
            "run_id": run.info.run_id,
            "run_name": run_name,
            "model_name": model.__class__.__name__,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
        }

## Run Multiple Models

At least three separate runs are required.  
This notebook compares:

- Linear Regression
- Ridge Regression (`alpha=0.1`)
- Ridge Regression (`alpha=1.0`)
- Lasso Regression (`alpha=0.01`)


In [ ]:
results = []

results.append(
    run_experiment(
        run_name="LinearRegression_baseline",
        model=LinearRegression(),
        params={"model_family": "linear_baseline"}
    )
)

results.append(
    run_experiment(
        run_name="Ridge_alpha_0_1",
        model=Ridge(alpha=0.1),
        params={"alpha": 0.1}
    )
)

results.append(
    run_experiment(
        run_name="Ridge_alpha_1_0",
        model=Ridge(alpha=1.0),
        params={"alpha": 1.0}
    )
)

results.append(
    run_experiment(
        run_name="Lasso_alpha_0_01",
        model=Lasso(alpha=0.01, max_iter=10000),
        params={"alpha": 0.01, "max_iter": 10000}
    )
)

results_df = pd.DataFrame(results).sort_values(by="RMSE", ascending=True).reset_index(drop=True)
results_df

## Save Run Summary

The run comparison table is also stored as a CSV artifact in the local project folder.


In [ ]:
summary_path = OUTPUT_DIR / "run_summary.csv"
results_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)

## Best Run Selection

The best model is selected using the **lowest RMSE** on the test set.


In [ ]:
best_row = results_df.iloc[0]

best_run_id = best_row["run_id"]
best_model_name = best_row["model_name"]

print("Best run ID :", best_run_id)
print("Best model  :", best_model_name)
print("Best RMSE   :", best_row["RMSE"])
print("Best MAE    :", best_row["MAE"])
print("Best R2     :", best_row["R2"])

## Bonus: Confidence Intervals with statsmodels

An OLS model is fitted on the transformed training data.
The notebook creates:
- coefficient confidence intervals,
- prediction confidence intervals for the test set.

Both files are saved locally and later logged to MLflow as artifacts.


In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_sm = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)
X_test_sm = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

alpha = 0.05  # 95% confidence interval

ols_model = sm.OLS(y_train, sm.add_constant(X_train_sm, has_constant="add")).fit()

coef_conf_intervals = ols_model.conf_int(alpha=alpha)
coef_conf_intervals.columns = ["lower_95", "upper_95"]

prediction_summary = ols_model.get_prediction(
    sm.add_constant(X_test_sm, has_constant="add")
).summary_frame(alpha=alpha)

coef_ci_path = OUTPUT_DIR / "conf_intervals_coefficients.csv"
pred_ci_path = OUTPUT_DIR / "conf_intervals_predictions.csv"

coef_conf_intervals.to_csv(coef_ci_path)
prediction_summary.to_csv(pred_ci_path)

display(coef_conf_intervals.head())
display(prediction_summary.head())

## Log Confidence Interval Artifacts to the Best Run

This step attaches the bonus artifacts to the selected best-performing run.


In [ ]:
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(str(coef_ci_path), artifact_path="confidence_intervals")
    mlflow.log_artifact(str(pred_ci_path), artifact_path="confidence_intervals")
    mlflow.log_artifact(str(summary_path), artifact_path="comparison")

print("Confidence interval artifacts logged to best run.")

## Register Best Model in the MLflow Model Registry

The best run is registered under the name:

- `BestRegressor_v1`


In [ ]:
registered_model_name = "BestRegressor_v1"
model_uri = f"runs:/{best_run_id}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

print("Registered model name   :", registered_model.name)
print("Registered model version:", registered_model.version)

## Move Model Version to Production

This notebook tries to move the registered model version to the `Production` stage.
If your local MLflow version shows a deprecation warning for stages, that is fine for this assignment.


In [ ]:
client = MlflowClient()

try:
    client.transition_model_version_stage(
        name=registered_model_name,
        version=registered_model.version,
        stage="Production"
    )
    print(f"Model {registered_model_name} version {registered_model.version} moved to Production.")
except Exception as e:
    print("Could not move model to Production stage:")
    print(e)

## Optional Alias

An alias is also assigned when supported by the installed MLflow version.


In [ ]:
try:
    client.set_registered_model_alias(
        name=registered_model_name,
        alias="champion",
        version=registered_model.version
    )
    print("Alias 'champion' set successfully.")
except Exception as e:
    print("Alias could not be set:")
    print(e)

## Verify Registered Model Loading

This final check loads the registered model from the local MLflow registry and runs a few sample predictions.


In [ ]:
try:
    loaded_model = mlflow.sklearn.load_model(
        model_uri=f"models:/{registered_model_name}/Production"
    )
except Exception:
    loaded_model = mlflow.sklearn.load_model(
        model_uri=f"models:/{registered_model_name}/{registered_model.version}"
    )

sample_pred = loaded_model.predict(X_test.iloc[:3])
print("Sample predictions:", sample_pred)

## What to Screenshot for `phase2.pdf`

Use the MLflow UI at `http://127.0.0.1:5000` and include:

1. **Experiment overview**
2. **Run comparison** (table or chart view)
3. **Registered model page**
4. **Short discussion**
   - why the best model was selected,
   - which metric was decisive,
   - what MAE, RMSE, and R² mean in this project.

This notebook already generates all required runs and artifacts for those screenshots.


## Conclusion

This notebook completes the full Phase 2 workflow:
- multiple tracked MLflow runs,
- logged parameters, metrics, artifacts, and tags,
- best-model comparison,
- model registration,
- optional confidence intervals for additional interpretability.

The registered model can now be used in **Phase 3** inside a local Streamlit application.
